In [1]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from models.model_DeepLOB import initiate_DeepLOB_model
from models.model_CNN_TE  import initiate_CNN_Transformer_model

from pathlib import Path
from data_loader import load_datasets

In [ ]:
base_path = Path(r"C:.\DeepLOB_TE\data")

x_train, y_train, x_val, y_val, x_test, y_test = load_datasets(base_path=base_path, timestamp_per_sample=100)

print(x_train.shape, y_train.shape)
print(x_val.shape, y_val.shape)
print(x_test.shape, y_test.shape)

(203720, 100, 40, 1) (203720, 3)
(50931, 100, 40, 1) (50931, 3)
(139488, 100, 40, 1) (139488, 3)


In [5]:
lookback_timestep = 100
feature_num = 40
conv_filter_num = 16
inception_num = 32
LSTM_num = 64
leaky_relu_alpha = 0.01

DeepLOB_model = initiate_DeepLOB_model(
    lookback_timestep=lookback_timestep,
    feature_num=feature_num,
    conv_filter_num=conv_filter_num,
    inception_num=inception_num,
    LSTM_num=LSTM_num,
    leaky_relu_alpha=leaky_relu_alpha,
    loss="categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01, epsilon=1),
    metrics=["accuracy"]
)

# DeepLOB_model.summary()

es = EarlyStopping(
    monitor="val_loss", 
    mode="auto", 
    patience=8,
    restore_best_weights=True,
    verbose=1 
)


history_DeepLOB = DeepLOB_model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=120, 
    batch_size=128, 
    callbacks=[es], 
    verbose=2
)


# DeepLOB_model.save("CNN_LSTM_model.keras")

Epoch 1/120
1592/1592 - 223s - 140ms/step - accuracy: 0.3489 - loss: 1.0967 - val_accuracy: 0.3165 - val_loss: 1.1046
Epoch 2/120
1592/1592 - 261s - 164ms/step - accuracy: 0.3494 - loss: 1.0960 - val_accuracy: 0.3136 - val_loss: 1.1038
Epoch 3/120
1592/1592 - 256s - 161ms/step - accuracy: 0.3572 - loss: 1.0918 - val_accuracy: 0.3700 - val_loss: 1.0965
Epoch 4/120
1592/1592 - 247s - 155ms/step - accuracy: 0.4208 - loss: 1.0405 - val_accuracy: 0.4182 - val_loss: 1.0777
Epoch 5/120
1592/1592 - 2555s - 2s/step - accuracy: 0.4935 - loss: 0.9329 - val_accuracy: 0.4723 - val_loss: 0.9787
Epoch 6/120
1592/1592 - 194s - 122ms/step - accuracy: 0.5193 - loss: 0.8970 - val_accuracy: 0.5187 - val_loss: 0.9079
Epoch 7/120
1592/1592 - 256s - 161ms/step - accuracy: 0.5302 - loss: 0.8802 - val_accuracy: 0.5233 - val_loss: 0.9028
Epoch 8/120
1592/1592 - 257s - 161ms/step - accuracy: 0.5365 - loss: 0.8724 - val_accuracy: 0.5258 - val_loss: 0.8881
Epoch 9/120
1592/1592 - 245s - 154ms/step - accuracy: 0.54

In [ ]:
model = initiate_CNN_Transformer_model(
    lookback_timestep=100,
    feature_num=40,
    conv_filter_num=16,
    inception_num=32,
    transformer_num_heads=4,
    transformer_key_dim=24,
    transformer_ff_dim=128,
    transformer_num_layers=2,
    dense_num=64,
    leaky_relu_alpha=0.01,
    dropout_rate=0.1,
    loss="categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    metrics=["accuracy"]
)

# model.summary()

es = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=8,
    restore_best_weights=True,
    verbose=1
)

rlr = ReduceLROnPlateau(
    monitor="val_loss",
    mode="min",
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    verbose=1
)

history = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=120,
    batch_size=128,
    callbacks=[es, rlr],
    verbose=2
)


# model.save("CNN_T_model.keras")


Epoch 1/120
1592/1592 - 530s - 333ms/step - accuracy: 0.4726 - loss: 0.9922 - val_accuracy: 0.5019 - val_loss: 0.9721 - learning_rate: 1.0000e-04
Epoch 2/120
1592/1592 - 526s - 330ms/step - accuracy: 0.5922 - loss: 0.8630 - val_accuracy: 0.5344 - val_loss: 0.9416 - learning_rate: 1.0000e-04
Epoch 3/120
1592/1592 - 542s - 340ms/step - accuracy: 0.6336 - loss: 0.8097 - val_accuracy: 0.5565 - val_loss: 0.9026 - learning_rate: 1.0000e-04
Epoch 4/120
1592/1592 - 491s - 308ms/step - accuracy: 0.6570 - loss: 0.7742 - val_accuracy: 0.5638 - val_loss: 0.8980 - learning_rate: 1.0000e-04
Epoch 5/120
1592/1592 - 501s - 315ms/step - accuracy: 0.6719 - loss: 0.7502 - val_accuracy: 0.5643 - val_loss: 0.8880 - learning_rate: 1.0000e-04
Epoch 6/120
1592/1592 - 539s - 339ms/step - accuracy: 0.6825 - loss: 0.7313 - val_accuracy: 0.5817 - val_loss: 0.8740 - learning_rate: 1.0000e-04
Epoch 7/120
1592/1592 - 508s - 319ms/step - accuracy: 0.6906 - loss: 0.7180 - val_accuracy: 0.5851 - val_loss: 0.8840 - lea

model 1: 67 epochs, 250s average

model 2: 57 epochs, 500s average